In [7]:
#The script collects regridded files saved in scratch, calculates extreme precipitation days using 

#00. libraries and dask client 
import time
import xarray as xr
import numpy as np
import pandas as pd
import glob
import datetime
import time
import os
import gc

import matplotlib
import xarray, matplotlib.pyplot as plt
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
import cartopy
import cartopy.crs as ccrs
# import matplotlib.pyplot as plt
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm



import dask 
import subprocess
import dask.array as da
from memory_profiler import profile
from filelock import FileLock
from dask.distributed import Client
from dask.utils import SerializableLock


from dask.diagnostics import ProgressBar

client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38799 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/38799/status,
Dashboard: /proxy/38799/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40763,Workers: 0
Dashboard: /proxy/38799/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40573,Total threads: 2
Dashboard: /proxy/40537/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:36725,


In [10]:
#2.0 define functions (modified version to deal with UKESM and NorESM data dimensions)

def load_dataset(
    var,
    domain,
    GCM,
    scenario,
    meG,
    provider,
    RCM,
    meR,
    tempRes,
    idir
):

    path_mask = (
        f"{IDIR}{var}_{domain}_{GCM}_{sc}_{meG}_"
        f"{provider}_{RCM}_{meR}_{tempRes}_*.nc"
    )

    files = glob.glob(path_mask)

    # --------------------------------------------------------
    # remove problematic CRS variables
    # --------------------------------------------------------

    def sanitize(ds):

        if "crs" in ds.dims:
            ds = ds.drop_dims("crs")

        if "crs" in ds.variables:
            ds = ds.drop_vars("crs")

        return ds

    # --------------------------------------------------------
    # open dataset
    # --------------------------------------------------------

    with dask.config.set(
        **{'array.slicing.split_large_chunks': False}
    ):

        ds = xr.open_mfdataset(
            files,
            preprocess=sanitize,
            combine='by_coords',
            chunks={'time': -1}
        )

    # --------------------------------------------------------
    # chunk ONLY data variables
    # --------------------------------------------------------

    for v in ds.data_vars:

        if {'time', 'lat', 'lon'}.issubset(ds[v].dims):

            ds[v] = ds[v].chunk({
                'time': -1,
                'lat': 'auto',
                'lon': 'auto'
            })

    return ds


def determine_univariate(
    data,
    out_prefix,
    thresh=0.99,
    ttype='q+'
):

    # --------------------------------------------------------
    # threshold
    # --------------------------------------------------------

    if 'q' in ttype:

        threshold = data.quantile(
            thresh,
            'time'
        )

    else:

        threshold = thresh

    # --------------------------------------------------------
    # exceedance
    # --------------------------------------------------------

    if '+' in ttype:

        univ_exceed = xr.where(
            data > threshold,
            1,
            0
        )

    else:

        univ_exceed = xr.where(
            data < threshold,
            1,
            0
        )

    out_file = (
        f"{out_prefix}_univ_exceed.nc"
    )

    threshold_out_file = (
        f"{out_prefix}_threshold.nc"
    )

    # --------------------------------------------------------
    # save
    # --------------------------------------------------------

    with FileLock(f"{out_file}.lock"):

        univ_exceed.to_netcdf(out_file)

    with FileLock(f"{threshold_out_file}.lock"):

        threshold.to_netcdf(
            threshold_out_file
        )


def exceedance_univariate(
    data,
    threshold_file,
    out_file,
    ttype='q+'
):

    threshold = xr.open_dataset(
        threshold_file
    )

    if '+' in ttype:

        univ_exceed = xr.where(
            data > threshold,
            1,
            0
        )

    else:

        univ_exceed = xr.where(
            data < threshold,
            1,
            0
        )

    with FileLock(f"{out_file}.lock"):

        univ_exceed.to_netcdf(out_file)

In [4]:
#2.1.1 iteration --ACCESS 
GCMs = ['ACCESS-ESM1-5'] 
RCMs = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
scenarios = ['historical','ssp126','ssp245','ssp370']
vars = ['tas','hurs','pr']

for var in vars:
    for GCM in GCMs:
        for RCM in RCMs:
            for sc in scenarios:
                meG = 'r6i1p1f1'
                meR = 'v1-r1'
                domain = 'AUS-18'
                provider = 'NSW-Government'
                tempRes = 'day'
                IDIR = '/scratch/dt6/xw9650/tmp/regridded/'
                OUTDIR = '/scratch/dt6/xw9650/tmp/'
                
                data = load_dataset(var, domain, GCM, sc, meG, provider, RCM, meR, tempRes, IDIR)

                prefix = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}"
                
                if sc == 'historical':
                    determine_univariate(data[var], prefix, thresh=0.99, ttype='q+')
                else:
                    threshold_file = f"{OUTDIR}{var}_{GCM}_{RCM}_historical_threshold.nc"
                    out_file = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}_univ_exceed.nc"
                    exceedance_univariate(data[var], threshold_file, out_file)


In [3]:
#2.1.2 iteration --EC-Earth3-Veg
GCMs = ['EC-Earth3-Veg']  
RCMs = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
scenarios = ['historical','ssp126', 'ssp245','ssp370']
vars = ['tas','hurs','pr']

for var in vars:
    for GCM in GCMs:
        for RCM in RCMs:
            for sc in scenarios:
                meG = 'r1i1p1f1'
                meR = 'v1-r1'
                domain = 'AUS-18'
                provider = 'NSW-Government'
                tempRes = 'day'
                IDIR = '/scratch/dt6/xw9650/tmp/regridded/'
                OUTDIR = '/scratch/dt6/xw9650/tmp/'
                
                data = load_dataset(var, domain, GCM, sc, meG, provider, RCM, meR, tempRes, IDIR)

                prefix = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}"
                
                if sc == 'historical':
                    determine_univariate(data[var], prefix, thresh=0.99, ttype='q+')
                else:
                    threshold_file = f"{OUTDIR}{var}_{GCM}_{RCM}_historical_threshold.nc"
                    out_file = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}_univ_exceed.nc"
                    exceedance_univariate(data[var], threshold_file, out_file)


In [5]:
#2.1.3 iteration --MPI-ESM1-2-HR
GCMs = ['MPI-ESM1-2-HR']  
RCMs = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
scenarios = ['historical','ssp126','ssp245','ssp370']
vars = ['tas','hurs','pr']

for var in vars:
    for GCM in GCMs:
        for RCM in RCMs:
            for sc in scenarios:
                meG = 'r1i1p1f1'
                meR = 'v1-r1'
                domain = 'AUS-18'
                provider = 'NSW-Government'
                tempRes = 'day'
                IDIR = '/scratch/dt6/xw9650/tmp/regridded/'
                OUTDIR = '/scratch/dt6/xw9650/tmp/'
                
                data = load_dataset(var, domain, GCM, sc, meG, provider, RCM, meR, tempRes, IDIR)

                prefix = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}"
                
                if sc == 'historical':
                    determine_univariate(data[var], prefix, thresh=0.99, ttype='q+')
                else:
                    threshold_file = f"{OUTDIR}{var}_{GCM}_{RCM}_historical_threshold.nc"
                    out_file = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}_univ_exceed.nc"
                    exceedance_univariate(data[var], threshold_file, out_file)


In [4]:
#2.1.4 iteration --NorESM2-MM
GCMs = ['NorESM2-MM']  # Add more GCMs here
RCMs = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
scenarios = ['historical','ssp126','ssp245','ssp370']
vars = ['tas', 'hurs','pr']

for var in vars:
    for GCM in GCMs:
        for RCM in RCMs:
            for sc in scenarios:
                meG = 'r1i1p1f1'
                meR = 'v1-r1'
                domain = 'AUS-18'
                provider = 'NSW-Government'
                tempRes = 'day'
                IDIR = '/scratch/dt6/xw9650/tmp/regridded/'
                OUTDIR = '/scratch/dt6/xw9650/tmp/'
                
                data = load_dataset(var, domain, GCM, sc, meG, provider, RCM, meR, tempRes, IDIR)

                prefix = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}"
                
                if sc == 'historical':
                    determine_univariate(data[var], prefix, thresh=0.99, ttype='q+')
                else:
                    threshold_file = f"{OUTDIR}{var}_{GCM}_{RCM}_historical_threshold.nc"
                    out_file = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}_univ_exceed.nc"
                    exceedance_univariate(data[var], threshold_file, out_file)


In [3]:
#2.1.5 iteration --UKESM1-0-LL
GCMs = ['UKESM1-0-LL']  
RCMs = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
scenarios = ['historical','ssp126','ssp245','ssp370']
vars = ['tas','hurs','pr']

for var in vars:
    for GCM in GCMs:
        for RCM in RCMs:
            for sc in scenarios:
                meG = 'r1i1p1f2'
                meR = 'v1-r1'
                domain = 'AUS-18'
                provider = 'NSW-Government'
                tempRes = 'day'
                IDIR = '/scratch/dt6/xw9650/tmp/regridded/'
                OUTDIR = '/scratch/dt6/xw9650/tmp/'
                
                data = load_dataset(var, domain, GCM, sc, meG, provider, RCM, meR, tempRes, IDIR)

                prefix = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}"
                
                if sc == 'historical':
                    determine_univariate(data[var], prefix, thresh=0.99, ttype='q+')
                else:
                    threshold_file = f"{OUTDIR}{var}_{GCM}_{RCM}_historical_threshold.nc"
                    out_file = f"{OUTDIR}{var}_{GCM}_{RCM}_{sc}_univ_exceed.nc"
                    exceedance_univariate(data[var], threshold_file, out_file)
